# Moshi Compression — Phase 4: Depformer Co-Adaptation

**Goal.** Teach the frozen depformer to decode SmolLM2's hidden distribution into real Mimi audio codes, fixing the `<0x14>` babble from Phase 3.

**Why this phase exists.** Phases 1–3 trained the Temporal Transformer (TT) and achieved cos_sim=0.90 with the teacher TT's hidden states. But the depformer was trained by Kyutai against the *original 7B TT's* hidden distribution. Cos_sim=0.90 leaves a 10% geometric gap, and the depformer — a narrow, high-precision 8-codebook decoder — cannot bridge that gap without retraining.

**Design.**
- **NO live teacher** (Phase-2 template): fast, low memory, teacher-off-GPU.
- **Freeze TT + adapters**: preserves the P3 cos_sim=0.90 gate.
- **Unfreeze depformer-family** (`depformer`, `depformer_in`, `depformer_emb`, `depformer_text_emb`, `linears`) — teacher init → adapts to student hidden.
- **Loss**: 8-codebook cross-entropy between `forward_depformer_training` logits and cached Mimi codes from `mhassann/moshi-cache-codes`.

**Layout.**
- `cuda:0` — student TT + adapters (frozen, no optimizer memory → stays small).
- `cuda:1` — emb/text_emb/out_norm/text_linear (frozen) + depformer-family (**trainable**). All PagedAdamW8bit state lives here.

**Entry criteria.**
- Phase-3 checkpoint `mhassann/moshi-p3-ckpt/ckpt_step_1000.pt` (6.56 GB, val cos_sim=0.9005).
- Frozen-heads dataset `tasfiatanha/moshi-frozen-heads` (initial depformer weights = teacher's).
- Codes cache `mhassann/moshi-cache-codes` (`codes.npy` int16 (60000,17,375)).
- Hidden cache `mhassann/moshi-cache-s{0..2}p{0..3}` (12 datasets).

**Exit criteria.**
- Depformer CE loss < 3.0 on val (teacher range ~1.5–2.0 on 2048-card codes).
- Audio output via ngrok actually sounds like speech (not `<0x14>` babble).
- TT hidden-state cos_sim stays at ~0.90 (sanity: we froze it, should be unchanged).


## Cell 1 — Global patches

In [3]:
import os, sys
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch._dynamo.config.disable = True
# sys.stdout.reconfigure(encoding="utf-8")
print("torch.compile disabled, expandable_segments enabled")
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))


torch.compile disabled, expandable_segments enabled
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


## Cell 2 — Environment verification

In [4]:
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

assert torch.cuda.device_count() >= 2, "Phase 4 needs dual GPU"
torch.cuda.set_device(0)
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
=== environment check PASSED ===


## Cell 3 — Installs

In [5]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "einops",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes
print(f"moshi from: {moshi.__file__}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("=== installs OK ===")

import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
transformers: 4.44.2
bitsandbytes: 0.49.2
=== installs OK ===
CUDAGraphed monkey-patched to no-op


## Cell 4 — Write smol_temporal.py

Same SmolTemporalTransformer as Phase 2/3. The TT is frozen in Phase 4; this file only defines its shape.

In [7]:
import pathlib

DST = pathlib.Path('/kaggle/working/moshi_repo/moshi/models/smol_temporal.py')

_SRC = """# moshi/models/smol_temporal.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import torch
import torch.nn as nn
import transformers
from ..modules.streaming import StreamingModule, State


@dataclass
class _SmolState(State):
    past_key_values: Optional[tuple] = field(default=None)

    def reset(self, reset_mask: torch.Tensor) -> None:
        super().reset(reset_mask)
        self.past_key_values = None


class SmolTemporalTransformer(StreamingModule[_SmolState]):
    def __init__(
        self,
        teacher_dim: int = 4096,
        student_dim: int = 2048,
        hf_name: str = 'HuggingFaceTB/SmolLM2-1.7B',
        rope_theta: float = 10_000.0,
        device: str = 'cuda:0',
        dtype: torch.dtype = torch.float16,
    ):
        super().__init__()
        self.teacher_dim = teacher_dim
        self.student_dim = student_dim

        cfg = transformers.AutoConfig.from_pretrained(hf_name)
        cfg.rope_theta = rope_theta
        cfg.use_cache = True
        cfg.attn_implementation = 'eager'
        self.backbone = transformers.AutoModel.from_pretrained(
            hf_name, config=cfg, torch_dtype=dtype,
        )
        if hasattr(self.backbone, 'embed_tokens'):
            self.backbone.embed_tokens = nn.Identity()

        self.in_adapter  = nn.Linear(teacher_dim, student_dim, bias=False)
        self.out_adapter = nn.Linear(student_dim, teacher_dim, bias=False)
        nn.init.normal_(self.in_adapter.weight,  std=1.0 / (teacher_dim ** 0.5))
        nn.init.normal_(self.out_adapter.weight, std=1.0 / (student_dim ** 0.5))

        self.to(device=device, dtype=dtype)

    def _init_streaming_state(self, batch_size: int) -> _SmolState:
        device = self.in_adapter.weight.device
        return _SmolState(batch_size=batch_size, device=device, past_key_values=None)

    def forward(
        self,
        x: torch.Tensor,
        cross_attention_src: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        assert cross_attention_src is None
        assert x.dim() == 3 and x.shape[-1] == self.teacher_dim

        x = x.to(self.in_adapter.weight.device)

        past_kv = (self._streaming_state.past_key_values
                   if self._streaming_state is not None else None)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            h = self.in_adapter(x)
            out = self.backbone(
                inputs_embeds=h,
                past_key_values=past_kv,
                use_cache=(self._streaming_state is not None),
                return_dict=True,
            )
            y = self.out_adapter(out.last_hidden_state)

        if self._streaming_state is not None:
            self._streaming_state.past_key_values = out.past_key_values

        return y

    def student_state_dict(self):
        return {
            'backbone':    self.backbone.state_dict(),
            'in_adapter':  self.in_adapter.state_dict(),
            'out_adapter': self.out_adapter.state_dict(),
        }

    def load_student_state_dict(self, sd: dict):
        self.backbone.load_state_dict(sd['backbone'])
        self.in_adapter.load_state_dict(sd['in_adapter'])
        self.out_adapter.load_state_dict(sd['out_adapter'])
"""

DST.write_text(_SRC)
print(f"Wrote {DST} ({DST.stat().st_size} bytes)")

import importlib, moshi.models
if hasattr(moshi.models, 'smol_temporal'):
    importlib.reload(moshi.models.smol_temporal)
from moshi.models.smol_temporal import SmolTemporalTransformer
print(f"SmolTemporalTransformer imported OK: {SmolTemporalTransformer}")
print("=== Cell 4 PASSED ===")


Wrote /kaggle/working/moshi_repo/moshi/models/smol_temporal.py (3255 bytes)
SmolTemporalTransformer imported OK: <class 'moshi.models.smol_temporal.SmolTemporalTransformer'>
=== Cell 4 PASSED ===


## Cell 5 — Open cache handles (codes + hidden)

Phase-4 training uses **codes** as CE targets (`codes.npy` from S14 cache). We also need **hidden** from the S2..S13 cache to drive the student's `forward_text` (student TT is frozen, so its output is deterministic per window — we still need to run it each step to produce `transformer_out` for the depformer).

In [8]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

WINDOWS_PER_PART = 5_000
TOTAL_WINDOWS    = 60_000
T_FRAMES         = 375
TEACHER_DIM      = 4096
N_CB             = 17
TOP_K            = 256
VAL_FRACTION     = 0.02
SEED             = 42

parts_hidden = []
parts_idx    = []
parts_val    = []

for shard in range(3):
    for part in range(4):
        for prefix in [
            f"/kaggle/input/moshi-cache-s{shard}p{part}",
            f"/kaggle/input/datasets/mhassann/moshi-cache-s{shard}p{part}",
        ]:
            import os
            if os.path.isdir(prefix):
                break
        parts_hidden.append(np.memmap(f"{prefix}/hidden.npy",
            dtype="float16", mode="r", shape=(5000, 375, 4096)))
        parts_idx.append(np.memmap(f"{prefix}/topk_idx.npy",
            dtype="int32",   mode="r", shape=(5000, 375, 256)))
        parts_val.append(np.memmap(f"{prefix}/topk_val.npy",
            dtype="float16", mode="r", shape=(5000, 375, 256)))

print(f"Opened {len(parts_hidden)} hidden memmap handles")

codes_path_candidates = [
    "/kaggle/input/moshi-cache-codes/codes.npy",
    "/kaggle/input/datasets/mhassann/moshi-cache-codes/codes.npy",
]
codes_path = None
for cp in codes_path_candidates:
    if os.path.exists(cp):
        codes_path = cp
        break
if codes_path is None:
    raise FileNotFoundError(f"codes.npy not found in: {codes_path_candidates}")

codes_mm = np.memmap(codes_path, dtype="int16", mode="r",
                     shape=(TOTAL_WINDOWS, N_CB, T_FRAMES))
print(f"Codes memmap: {codes_mm.shape} dtype={codes_mm.dtype}")

# Same split seed as Phases 1/2/3 — identical train/val partition.
rng = random.Random(SEED)
all_indices = list(range(TOTAL_WINDOWS))
rng.shuffle(all_indices)
n_val = max(1, int(TOTAL_WINDOWS * VAL_FRACTION))
val_indices = set(all_indices[:n_val])
train_indices = [i for i in range(TOTAL_WINDOWS) if i not in val_indices]
print(f"Train: {len(train_indices)}, Val: {n_val}")


class CacheDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        part_i, local_i = divmod(i, WINDOWS_PER_PART)

        c = np.array(codes_mm[i], copy=True)
        return {
            "codes": torch.from_numpy(c.astype(np.int64)),
        }


train_ds = CacheDataset(train_indices)
val_ds   = CacheDataset(list(val_indices))

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=0, pin_memory=True)

batch = train_ds[0]
print(f"Sample: codes {batch['codes'].shape} {batch['codes'].dtype}")
print("=== Cell 5 PASSED ===")


Opened 12 hidden memmap handles
Codes memmap: (60000, 17, 375) dtype=int16
Train: 58800, Val: 1200
Sample: codes torch.Size([17, 375]) torch.int64
=== Cell 5 PASSED ===


## Cell 6 — Build student (P3 init for TT, frozen-heads init for depformer)

Student layout for Phase 4:
- `cuda:0`: TT + adapters (frozen, `requires_grad=False`, no optimizer memory).
- `cuda:1`: all frozen heads + all trainable depformer-family.

Init sources:
- **TT weights** from `mhassann/moshi-p3-ckpt/ckpt_step_1000.pt` (student_backbone + in_adapter + out_adapter).
- **Depformer-family** from `tasfiatanha/moshi-frozen-heads` (teacher's depformer — the starting point for co-adaptation).
- **Other frozen heads** (emb, text_emb, out_norm, text_linear) also from frozen-heads dataset.

In [9]:
import torch, pathlib, gc, ctypes
from moshi.models.loaders import CheckpointInfo
from moshi.models.smol_temporal import SmolTemporalTransformer
import pickle as _pickle

# Arena trim — critical on Kaggle to avoid 30GB RSS session kills.
try:
    _libc = ctypes.CDLL("libc.so.6")
    def _trim_ram():
        try: _libc.malloc_trim(0)
        except Exception: pass
except OSError:
    def _trim_ram(): pass

# numpy 2.x pickle compat
class _NumpyCompatUnpickler(_pickle.Unpickler):
    _REMAP = {
        "numpy.core.multiarray": "numpy._core.multiarray",
        "numpy.core.numeric":    "numpy._core.numeric",
        "numpy.core.umath":      "numpy._core.umath",
        "numpy.core":            "numpy._core",
    }
    def find_class(self, module, name):
        return super().find_class(self._REMAP.get(module, module), name)

class _NpPickle:
    Unpickler        = _NumpyCompatUnpickler
    loads            = staticmethod(_pickle.loads)
    load             = staticmethod(_pickle.load)
    dump             = staticmethod(_pickle.dump)
    dumps            = staticmethod(_pickle.dumps)
    HIGHEST_PROTOCOL = _pickle.HIGHEST_PROTOCOL
    DEFAULT_PROTOCOL = _pickle.DEFAULT_PROTOCOL
    PickleError      = _pickle.PickleError
    UnpicklingError  = _pickle.UnpicklingError

def _torch_load(path, **kw):
    kw.setdefault("weights_only", False)
    kw.setdefault("pickle_module", _NpPickle)
    return torch.load(path, **kw)


def _host_rss_gb():
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    except Exception:
        return -1.0

def _mem(tag):
    torch.cuda.synchronize()
    parts = [f"host={_host_rss_gb():5.1f}GB"]
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        parts.append(f"cuda:{i} {(total-free)/1e9:4.1f}/{total/1e9:4.1f}GB")
    print(f"[MEM {tag:32s}] " + "  ".join(parts))


gc.collect(); _trim_ram()
_mem("start of Cell 6")

from moshi.models.lm import LMModel

print("Building student shell ...")
student_lm = LMModel(
    dim=4096, num_heads=32, num_layers=32, hidden_scale=4.125,
    gating="silu", norm="rms_norm_f32", positional_embedding="rope",
    context=3000, n_q=16, dep_q=8, card=2048, text_card=32000,
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    depformer_dim=1024, depformer_dim_feedforward=4224,
    depformer_num_heads=16, depformer_num_layers=6,
    depformer_multi_linear=True, depformer_weights_per_step=True,
    depformer_context=8, depformer_pos_emb="none",
    existing_text_padding_id=3,
).to(dtype=torch.float16)
_mem("after LMModel shell (CPU)")

# Move everything to cuda:1 EXCEPT TT (we'll swap TT next).
# All heads + depformer-family on cuda:1.
ALL_HEADS_CUDA1 = ["emb", "text_emb", "out_norm", "text_linear",
                   "depformer_in", "depformer_emb", "depformer_text_emb",
                   "depformer", "linears"]
for attr in ALL_HEADS_CUDA1:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:1")
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after heads -> cuda:1")

print("Replacing Helium TT with SmolLM2-1.7B (direct to cuda:0) ...")
smol_tt = SmolTemporalTransformer(
    teacher_dim=4096, student_dim=2048,
    hf_name="HuggingFaceTB/SmolLM2-1.7B",
    rope_theta=10_000.0,
    device="cuda:0", dtype=torch.float16,
)
old_tt = student_lm.transformer
student_lm.transformer = smol_tt
del old_tt
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after smol_tt swap")

# Phase 4 trainability:
#   TT + adapters    -> FROZEN (preserve P3 cos_sim=0.90 gate)
#   depformer-family -> TRAINABLE (these learn to decode student hidden)
#   other heads      -> FROZEN
TRAINABLE_ATTRS = {"depformer", "depformer_in", "depformer_emb",
                   "depformer_text_emb", "linears"}
for name, p in student_lm.named_parameters():
    top = name.split(".")[0]
    p.requires_grad_(top in TRAINABLE_ATTRS)

# Frozen heads — load each .pt onto its final GPU.
FROZEN_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-frozen-heads"),
    pathlib.Path("/kaggle/input/moshi-frozen-heads"),
]
frozen_dir = None
for p in FROZEN_CANDIDATES:
    if p.exists():
        frozen_dir = p
        break
if frozen_dir is None:
    raise FileNotFoundError(f"Frozen heads not found: {FROZEN_CANDIDATES}")

# OPTION 5 DIAGNOSTIC: random-init depformer (isolates "basin lock" from "budget limit")
# - False: teacher-init depformer-family (standard co-adaptation).
# - True:  load only the TRULY frozen heads (emb, text_emb, out_norm, text_linear),
#          leave depformer-family at PyTorch default init (essentially random).
RANDOM_INIT_DEPFORMER = True

HEADS_TO_LOAD_FROM_DISK = (
    ["emb", "text_emb", "out_norm", "text_linear"]
    if RANDOM_INIT_DEPFORMER
    else ALL_HEADS_CUDA1
)
print(f"Loading {len(HEADS_TO_LOAD_FROM_DISK)} head modules from {frozen_dir} (direct-to-cuda:1) ...")
print(f"  (RANDOM_INIT_DEPFORMER={RANDOM_INIT_DEPFORMER}  "
      f"=> depformer-family {'at default init' if RANDOM_INIT_DEPFORMER else 'from teacher'})")
for name in HEADS_TO_LOAD_FROM_DISK:
    pt_file = frozen_dir / f"{name}.pt"
    if not pt_file.exists():
        continue
    sd = torch.load(pt_file, map_location="cuda:1", weights_only=True)
    getattr(student_lm, name).load_state_dict(sd)
    del sd
    gc.collect(); _trim_ram()

# Re-init the depformer-family explicitly under RANDOM_INIT_DEPFORMER so the
# starting CE is predictable (~ln(2048) = 7.62 for a reasonable init, not
# whatever leaked in from the shell construction).
if RANDOM_INIT_DEPFORMER:
    import torch.nn as nn
    def _reinit(mod):
        for m in mod.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=1.0 / (m.in_features ** 0.5))
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=1.0 / (m.embedding_dim ** 0.5))
    _reinit(student_lm.depformer)
    _reinit(student_lm.depformer_in)
    _reinit(student_lm.depformer_emb)
    _reinit(student_lm.depformer_text_emb)
    _reinit(student_lm.linears)
    print("  depformer-family re-initialized from scratch (random)")

_mem("after head load (+ optional depformer reinit)")

n_train = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
n_froz  = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
print(f"Trainable : {n_train/1e6:.1f} M  ({n_train/1e9:.3f} B)")
print(f"Frozen    : {n_froz/1e6:.1f} M  ({n_froz/1e9:.3f} B)")

# Cross-device forward_text patch — same pattern as Phase 3.
import types as _types

def _fixed_forward_text(self, sequence, sum_condition=None, cross_attention_src=None):
    B, K, S = sequence.shape
    assert K == self.num_codebooks

    emb_device = next(self.emb[0].parameters()).device
    input_sequence = sequence.to(emb_device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    if sum_condition is not None:
        input_ = input_ + sum_condition.to(input_)
    if cross_attention_src is not None:
        cross_attention_src = cross_attention_src.to(input_)

    tt_device = next(self.transformer.parameters()).device
    transformer_out = self.transformer(
        input_.to(tt_device), cross_attention_src=cross_attention_src,
    )

    if self.out_norm:
        on_device = next(self.out_norm.parameters()).device
        transformer_out = self.out_norm(transformer_out.to(on_device))

    tl_device = next(self.text_linear.parameters()).device
    text_logits = self.text_linear(transformer_out.to(tl_device))
    text_logits = text_logits[:, None]
    return transformer_out, text_logits

student_lm.forward_text = _types.MethodType(_fixed_forward_text, student_lm)

# Cross-device forward_depformer_training patch.
# depformer_in / depformer_emb / depformer_text_emb / depformer / linears all on cuda:1;
# we only need transformer_out (from cuda:1 after out_norm) to stay on cuda:1.
def _fixed_forward_depformer_training(self, sequence, transformer_out):
    assert self.depformer_text_emb is not None
    assert self.depformer_emb is not None
    assert self.depformer is not None

    # Align to cuda:1 (where the depformer lives).
    dep_device = next(self.depformer.parameters()).device
    sequence = sequence.to(dep_device)
    transformer_out = transformer_out.to(dep_device)

    B, K, T = sequence.shape
    Ka = self.dep_q
    assert K == self.num_codebooks, f"Codebooks for Depformer training: expected {self.num_codebooks}, got {K}"

    depformer_inputs = []
    for cb_index in range(Ka):
        if self.depformer_multi_linear:
            linear_index = cb_index
            if self.depformer_weights_per_step_schedule is not None:
                linear_index = self.depformer_weights_per_step_schedule[cb_index]
            transformer_in = self.depformer_in[linear_index](transformer_out)
        else:
            transformer_in = self.depformer_in[0](transformer_out)
        if cb_index == 0:
            token_in = self.depformer_text_emb(sequence[:, 0])
        else:
            token_in = self.depformer_emb[cb_index - 1](sequence[:, cb_index + self.audio_offset - 1])
        depformer_inputs.append(token_in + transformer_in)
    depformer_input = torch.stack(depformer_inputs, 2)
    depformer_input = depformer_input.view(B * T, Ka, -1)
    depformer_output = self.depformer(depformer_input)
    all_logits = []
    for cb_index in range(Ka):
        logits = self.linears[cb_index](self.depformer_norms[cb_index](depformer_output[:, cb_index]))
        all_logits.append(logits.view(B, T, -1))
    logits = torch.stack(all_logits, 1)
    return logits  # [B, Ka, T, card=2048]

student_lm.forward_depformer_training = _types.MethodType(
    _fixed_forward_depformer_training, student_lm)

gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("end of Cell 6")
print("forward_text + forward_depformer_training patched (cross-device)")
print("=== Cell 6 PASSED ===")


[MEM start of Cell 6                 ] host=  0.9GB  cuda:0  0.1/15.6GB  cuda:1  0.1/15.6GB
Building student shell ...
[MEM after LMModel shell (CPU)       ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1  0.1/15.6GB
[MEM after heads -> cuda:1           ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1  2.3/15.6GB
Replacing Helium TT with SmolLM2-1.7B (direct to cuda:0) ...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

[MEM after smol_tt swap              ] host= 29.5GB  cuda:0  3.4/15.6GB  cuda:1  2.3/15.6GB
Loading 4 head modules from /kaggle/input/datasets/tasfiatanha/moshi-frozen-heads (direct-to-cuda:1) ...
  (RANDOM_INIT_DEPFORMER=True  => depformer-family at default init)
  depformer-family re-initialized from scratch (random)
[MEM after head load (+ optional depformer reinit)] host= 29.5GB  cuda:0  3.4/15.6GB  cuda:1  2.6/15.6GB
Trainable : 714.4 M  (0.714 B)
Frozen    : 2023.9 M  (2.024 B)
[MEM end of Cell 6                   ] host= 29.5GB  cuda:0  3.4/15.6GB  cuda:1  2.3/15.6GB
forward_text + forward_depformer_training patched (cross-device)
=== Cell 6 PASSED ===


## Cell 7 — Optimizer, scheduler, gradient checkpointing

In [10]:
import bitsandbytes as bnb
import math

# OPTION 5 DIAGNOSTIC CONFIG
#   Purpose: answer "is Phase 4 failing because of bad init, or bad budget?"
#   Outcome we care about: CE trajectory over first 500 steps from RANDOM init.
#     - If CE drops meaningfully below 7.62 (ln(2048)) -> init was the problem.
#     - If CE stays at 7+ -> 10% cos_sim gap is irreducible for this budget.
GRAD_ACCUM   = 4
LR           = 3e-4        # higher LR OK from random init (no weights to preserve)
LR_MIN       = 3e-5
WARMUP_STEPS = 50
MAX_STEPS    = 500         # SHORT diagnostic; we only need the trajectory shape
MAX_NORM     = 1.0

trainable_params = [p for p in student_lm.parameters() if p.requires_grad]
assert len(trainable_params) > 0, "No trainable params!"

optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=LR, weight_decay=0.01)

class _NoOpScaler:
    def scale(self, loss):   return loss
    def unscale_(self, opt): pass
    def step(self, opt):     opt.step()
    def update(self):        pass
    def get_scale(self):     return 1
    def state_dict(self):    return {}
    def load_state_dict(self, sd): pass
scaler = _NoOpScaler()

# Student TT is FROZEN → no gradient checkpointing on it (no grads flow).
# We still need `.eval()` on the backbone to disable dropout during the
# forward that produces transformer_out.
backbone = student_lm.transformer.backbone
backbone.config.use_cache = False
backbone.eval()
student_lm.transformer.eval()
# Depformer IS trainable → enable its train mode.
student_lm.depformer.train()
student_lm.train()  # top-level; per-submodule eval/train overrides above.

# Make sure TT + adapters are actually frozen (belt-and-suspenders):
for p in student_lm.transformer.parameters():
    p.requires_grad_(False)

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return max(LR_MIN / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

print(f"Optimizer: PagedAdamW8bit, lr={LR}, wd=0.01")
print(f"Scheduler: warmup {WARMUP_STEPS}, cosine -> {LR_MIN}")
print(f"Grad accum: {GRAD_ACCUM}, max_norm: {MAX_NORM}")
print(f"Max steps this session: {MAX_STEPS}")
n_train = sum(p.numel() for p in trainable_params)
print(f"Trainable params: {n_train/1e6:.1f} M (depformer + in/emb/linears)")
print("=== Cell 7 PASSED ===")


Optimizer: PagedAdamW8bit, lr=0.0003, wd=0.01
Scheduler: warmup 50, cosine -> 3e-05
Grad accum: 4, max_norm: 1.0
Max steps this session: 500
Trainable params: 714.4 M (depformer + in/emb/linears)
=== Cell 7 PASSED ===


## Cell 8 — Resume TT weights from P3, depformer from P4 (if any)

Priority:
1. Phase-4 checkpoint (full resume: depformer + optimizer state) — for continuation across sessions.
2. Phase-3 checkpoint — restores TT only (fresh depformer from frozen-heads init, fresh optimizer).

In [11]:
import pathlib, glob, json, random, os
import numpy as np

P4_CANDIDATES = [
    "/kaggle/working",
    "/kaggle/input/datasets/mhassann/moshi-p4-ckpt",
    "/kaggle/input/moshi-p4-ckpt",
]
P3_CANDIDATES = [
    "/kaggle/input/datasets/mhassann/moshi-p3-ckpt",
    "/kaggle/input/moshi-p3-ckpt",
]

start_step = 0
total_wall = 0.0
ckpt_loaded = False
ckpt_phase  = None

# Phase-4 full resume (includes depformer + optimizer)
for ckpt_dir in P4_CANDIDATES:
    if not os.path.isdir(ckpt_dir):
        continue
    ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
    if not ckpts:
        continue
    latest = ckpts[-1]
    print(f"Found Phase-4 checkpoint: {latest}")
    ckpt = _torch_load(latest, map_location="cpu")
    if ckpt.get("phase") != "P4":
        print(f"  skipping — phase={ckpt.get('phase')} (not P4)")
        del ckpt
        continue

    # TT + adapters (frozen, but we still need the exact P3 weights)
    smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
    smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
    smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
    # Depformer-family (trainable)
    student_lm.depformer.load_state_dict(ckpt["depformer"])
    student_lm.depformer_in.load_state_dict(ckpt["depformer_in"])
    student_lm.depformer_emb.load_state_dict(ckpt["depformer_emb"])
    student_lm.depformer_text_emb.load_state_dict(ckpt["depformer_text_emb"])
    student_lm.linears.load_state_dict(ckpt["linears"])
    print("  TT + depformer-family restored")

    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print("  optimizer + scheduler + scaler restored")

    if "torch_rng" in ckpt:  torch.set_rng_state(ckpt["torch_rng"])
    if "cuda_rng" in ckpt:   torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    if "numpy_rng" in ckpt:  np.random.set_state(ckpt["numpy_rng"])
    if "python_rng" in ckpt: random.setstate(ckpt["python_rng"])

    start_step = ckpt.get("step", 0)
    total_wall = ckpt.get("wall_seconds", 0.0)
    ckpt_loaded = True
    ckpt_phase = "P4"
    del ckpt
    torch.cuda.empty_cache()
    print(f"  Resuming Phase-4 from step {start_step}")
    break

# Fallback: init TT from Phase-3 (depformer stays as frozen-heads init)
if not ckpt_loaded:
    for ckpt_dir in P3_CANDIDATES:
        if not os.path.isdir(ckpt_dir):
            continue
        ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
        if not ckpts:
            continue
        latest = ckpts[-1]
        print(f"Initializing TT from Phase-3 checkpoint: {latest}")
        ckpt = _torch_load(latest, map_location="cpu")

        smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
        smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
        smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
        print("  Phase-3 TT weights loaded.")
        print("  Depformer-family: whatever Cell 6 produced "
              "(teacher init if RANDOM_INIT_DEPFORMER=False, random otherwise).")
        print("  Optimizer: fresh.")
        ckpt_loaded = True
        ckpt_phase = "P3-init"
        del ckpt
        torch.cuda.empty_cache()
        break

if not ckpt_loaded:
    raise RuntimeError(
        "No checkpoint found — Phase 4 requires either a Phase-3 final "
        "checkpoint or a prior Phase-4 checkpoint to resume from."
    )

print(f"Start step: {start_step}, phase init: {ckpt_phase}")
print("=== Cell 8 PASSED ===")


Initializing TT from Phase-3 checkpoint: /kaggle/input/datasets/mhassann/moshi-p3-ckpt/ckpt_step_1056.pt
  Phase-3 TT weights loaded.
  Depformer-family: whatever Cell 6 produced (teacher init if RANDOM_INIT_DEPFORMER=False, random otherwise).
  Optimizer: fresh.
Start step: 0, phase init: P3-init
=== Cell 8 PASSED ===


## Cell 9 — Loss function (8-codebook CE)

Moshi's `forward_depformer_training` returns logits `[B, dep_q=8, T, card=2048]`.
Targets are the Mimi audio codebooks `codes[:, audio_offset:audio_offset+dep_q, :]` (audio_offset=1, so codebooks 1..8).

We compute CE per codebook, per position, then average.

In [12]:
import torch
import torch.nn.functional as F

AUDIO_OFFSET = 1  # codes[:, 0] is text; codes[:, 1:9] are the 8 audio codebooks
DEP_Q        = 8
CARD         = 2048


def depformer_ce_loss(logits, target_codes):
    """
    logits:        [B, Ka=8, T, card=2048]
    target_codes:  [B, 17, T]  — full codebook stack, we slice [:, 1:9, :]
    Returns: scalar CE, averaged over (B, Ka, T).
    """
    B, Ka, T, C = logits.shape
    assert Ka == DEP_Q and C == CARD

    # Align device — targets probably on cuda:1 already, but defensively:
    targets = target_codes[:, AUDIO_OFFSET:AUDIO_OFFSET + DEP_Q, :].long().to(logits.device)
    # targets: [B, Ka, T] with values in [0, 2048)

    # Flatten to [(B*Ka*T), C] vs [(B*Ka*T)]
    logits_flat = logits.reshape(-1, C).float()
    target_flat = targets.reshape(-1)

    # Clamp targets to valid range defensively — any -1 or out-of-range
    # in cache would crash CE otherwise.
    valid_mask = (target_flat >= 0) & (target_flat < C)
    if not valid_mask.all():
        logits_flat = logits_flat[valid_mask]
        target_flat = target_flat[valid_mask]

    return F.cross_entropy(logits_flat, target_flat)


def per_codebook_ce(logits, target_codes):
    """Same loss but returned per-codebook, for diagnostics."""
    B, Ka, T, C = logits.shape
    targets = target_codes[:, AUDIO_OFFSET:AUDIO_OFFSET + DEP_Q, :].long().to(logits.device)
    out = []
    for k in range(Ka):
        l = logits[:, k].reshape(-1, C).float()
        t = targets[:, k].reshape(-1)
        m = (t >= 0) & (t < C)
        if m.any():
            out.append(F.cross_entropy(l[m], t[m]).item())
        else:
            out.append(float("nan"))
    return out


print("Loss functions defined: depformer_ce_loss, per_codebook_ce")
print(f"AUDIO_OFFSET={AUDIO_OFFSET}, DEP_Q={DEP_Q}, CARD={CARD}")
print("=== Cell 9 PASSED ===")


Loss functions defined: depformer_ce_loss, per_codebook_ce
AUDIO_OFFSET=1, DEP_Q=8, CARD=2048
=== Cell 9 PASSED ===


## Cell 10 — Validation function

Reports:
- `val_depformer_ce` (training metric).
- `val_depformer_ce_cb` (per-codebook list) — useful to spot bad codebooks.

We also run `forward_text` once per window to confirm TT hidden states are still producing reasonable output (cos_sim proxy not computed here — TT is frozen so the gate should be unchanged from P3).

In [13]:
@torch.no_grad()
def validate(model, val_loader, device="cuda:0"):
    model.transformer.eval()
    model.depformer.eval()
    total_ce = 0.0
    total_cb = [0.0] * DEP_Q
    n = 0
    T_CHUNK = 125
    for batch in val_loader:
        codes_b = batch["codes"].to(device)  # [1, 17, 375] on cuda:0 for TT input
        T_total = codes_b.shape[-1]
        n_chunks = (T_total + T_CHUNK - 1) // T_CHUNK
        chunk_ce = 0.0
        chunk_cb = [0.0] * DEP_Q
        for c_i in range(n_chunks):
            a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T_total)
            codes_c = codes_b[..., a:b]

            transformer_out, _ = model.forward_text(codes_c)
            # transformer_out on cuda:1 (after out_norm hop)
            logits = model.forward_depformer_training(codes_c, transformer_out)
            ce = depformer_ce_loss(logits, codes_c).item()
            cb = per_codebook_ce(logits, codes_c)
            chunk_ce += ce * (b - a) / T_total
            for k in range(DEP_Q):
                chunk_cb[k] += cb[k] * (b - a) / T_total
        total_ce += chunk_ce
        for k in range(DEP_Q):
            total_cb[k] += chunk_cb[k]
        n += 1

    model.depformer.train()
    return {
        "val_depformer_ce": total_ce / n,
        "val_depformer_ce_cb": [c / n for c in total_cb],
    }


print("validate() defined")
print("=== Cell 10 PASSED ===")


validate() defined
=== Cell 10 PASSED ===


## Cell 11 — Training loop

Per micro-step:
1. TT forward (no_grad, since TT is frozen) → `transformer_out` on cuda:1.
2. Depformer forward (with grad) → logits.
3. CE loss vs `codes[:, 1:9, :]`.
4. Backward + step every `GRAD_ACCUM` micro-steps.

**Gradient only flows through the depformer-family.** The TT forward is wrapped in `torch.no_grad()` to save memory (no activation saving for the frozen 1.7B SmolLM2).

In [14]:
import time, json, pathlib

OUT_DIR = pathlib.Path("/kaggle/working")
LOG_PATH = OUT_DIR / "train_log.jsonl"

device = "cuda:0"  # codes land on cuda:0 first (for TT input)

log_file = open(LOG_PATH, "a")

step = start_step
micro_step = 0
accum_loss = 0.0
t_session = time.time()

print(f"Starting Phase-4 training from step {step}, max {MAX_STEPS} steps this session")
print(f"Grad accum = {GRAD_ACCUM}")
print(f"Loss: depformer_ce (8-codebook avg)")
print()

optimizer.zero_grad(set_to_none=True)

data_iter = iter(train_loader)

while step < start_step + MAX_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    codes_b = batch["codes"].to(device)

    T = codes_b.shape[-1]
    T_CHUNK = 125
    micro = 0.0
    n_chunks = (T + T_CHUNK - 1) // T_CHUNK
    for c_i in range(n_chunks):
        a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T)
        codes_c = codes_b[..., a:b]

        # TT forward — frozen, no grad (saves 1.7B of activations).
        with torch.no_grad():
            transformer_out, _ = student_lm.forward_text(codes_c)
            # transformer_out now on cuda:1 (after out_norm hop)

        # Depformer forward — WITH grad.
        logits = student_lm.forward_depformer_training(codes_c, transformer_out)
        loss = depformer_ce_loss(logits, codes_c)

        chunk_frac = (b - a) / T
        scaled = loss * chunk_frac / GRAD_ACCUM
        scaler.scale(scaled).backward()
        micro += loss.item() * chunk_frac

    accum_loss += micro
    micro_step += 1

    if micro_step % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        step += 1
        wall = time.time() - t_session + total_wall
        avg = accum_loss / GRAD_ACCUM

        gn = float(grad_norm)
        gn_str = f"{gn:.3f}" if gn < 1e6 else "inf"

        row = {
            "step": step,
            "phase": "P4",
            "loss_depformer_ce": round(avg, 5),
            "grad_norm": round(gn, 4) if gn < 1e6 else None,
            "lr": round(scheduler.get_last_lr()[0], 7),
            "wall_s": round(wall, 1),
        }
        log_file.write(json.dumps(row) + "\n")
        log_file.flush()

        if step % 20 == 0:
            print(f"step {step:5d}  ce={avg:.4f}  gn={gn_str}  "
                  f"lr={row['lr']:.2e}  wall={wall:.0f}s")

        accum_loss = 0.0

        if step % 200 == 0:
            val_result = validate(student_lm, val_loader, device)
            cb_str = " ".join(f"{x:.2f}" for x in val_result["val_depformer_ce_cb"])
            print(f"  VAL step {step}: ce={val_result['val_depformer_ce']:.4f}  "
                  f"per_cb=[{cb_str}]")
            val_row = {"step": step, "phase": "P4", "type": "val",
                       **val_result, "wall_s": round(wall, 1)}
            log_file.write(json.dumps(val_row) + "\n")
            log_file.flush()
            student_lm.depformer.train()

            if val_result["val_depformer_ce"] < 3.0:
                print(f"  *** Phase-5 gate MET: val_depformer_ce = "
                      f"{val_result['val_depformer_ce']:.4f} < 3.0 ***")

        if step % 500 == 0:
            ckpt_path = OUT_DIR / f"ckpt_step_{step}.pt"
            ckpt = {
                "step": step,
                "phase": "P4",
                "wall_seconds": round(wall, 1),
                # TT weights (frozen, but save for single-file resume)
                "student_backbone": smol_tt.backbone.state_dict(),
                "in_adapter":       smol_tt.in_adapter.state_dict(),
                "out_adapter":      smol_tt.out_adapter.state_dict(),
                # Depformer-family (trainable)
                "depformer":          student_lm.depformer.state_dict(),
                "depformer_in":       student_lm.depformer_in.state_dict(),
                "depformer_emb":      student_lm.depformer_emb.state_dict(),
                "depformer_text_emb": student_lm.depformer_text_emb.state_dict(),
                "linears":            student_lm.linears.state_dict(),
                # Training state
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler":    scaler.state_dict(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng":  torch.cuda.get_rng_state_all(),
                "numpy_rng": np.random.get_state(),
                "python_rng": random.getstate(),
                "torch_version": torch.__version__,
            }
            torch.save(ckpt, ckpt_path)
            size_gb = ckpt_path.stat().st_size / 1e9
            print(f"  CKPT saved: {ckpt_path.name} ({size_gb:.2f} GB)")
            del ckpt

log_file.close()

final_wall = time.time() - t_session + total_wall
print(f"\nTraining done. Final step: {step}, wall: {final_wall:.0f}s")

for i in range(2):
    free, total_ = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: free {free/1e9:.2f} / {total_/1e9:.2f} GB")

val_final = validate(student_lm, val_loader, device)
cb_str = " ".join(f"{x:.2f}" for x in val_final["val_depformer_ce_cb"])
print(f"Final VAL: ce={val_final['val_depformer_ce']:.4f}  per_cb=[{cb_str}]")
print("=== Cell 11 PASSED ===")


Starting Phase-4 training from step 0, max 500 steps this session
Grad accum = 4
Loss: depformer_ce (8-codebook avg)

step    20  ce=9.4535  gn=15.227  lr=1.20e-04  wall=33s
step    40  ce=11.7053  gn=20.953  lr=2.40e-04  wall=64s
step    60  ce=12.8753  gn=19.875  lr=3.00e-04  wall=95s
step    80  ce=12.5846  gn=26.047  lr=2.97e-04  wall=126s
step   100  ce=12.8370  gn=23.625  lr=2.91e-04  wall=157s
step   120  ce=12.6505  gn=27.969  lr=2.82e-04  wall=188s
step   140  ce=12.0045  gn=22.734  lr=2.71e-04  wall=218s
step   160  ce=10.0542  gn=10.531  lr=2.58e-04  wall=249s
step   180  ce=9.8383  gn=9.711  lr=2.42e-04  wall=280s
step   200  ce=9.5931  gn=13.133  lr=2.25e-04  wall=311s
  VAL step 200: ce=9.4031  per_cb=[7.95 8.57 8.50 9.03 8.08 11.91 10.44 10.74]
step   220  ce=8.8147  gn=6.836  lr=2.06e-04  wall=536s
step   240  ce=7.8284  gn=4.988  lr=1.86e-04  wall=567s
step   260  ce=7.6691  gn=4.441  lr=1.66e-04  wall=598s
step   280  ce=7.4991  gn=5.398  lr=1.45e-04  wall=629s
step  

## Cell 12 — Final checkpoint + push to Kaggle

In [ ]:
import subprocess, json, pathlib, os

OUT_DIR = pathlib.Path("/kaggle/working")
username = os.environ.get("KAGGLE_USERNAME", "mhassann")

final_ckpt = OUT_DIR / f"ckpt_step_{step}.pt"
if not final_ckpt.exists():
    ckpt = {
        "step": step,
        "phase": "P4",
        "wall_seconds": round(time.time() - t_session + total_wall, 1),
        "student_backbone": smol_tt.backbone.state_dict(),
        "in_adapter":       smol_tt.in_adapter.state_dict(),
        "out_adapter":      smol_tt.out_adapter.state_dict(),
        "depformer":          student_lm.depformer.state_dict(),
        "depformer_in":       student_lm.depformer_in.state_dict(),
        "depformer_emb":      student_lm.depformer_emb.state_dict(),
        "depformer_text_emb": student_lm.depformer_text_emb.state_dict(),
        "linears":            student_lm.linears.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler":    scaler.state_dict(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng":  torch.cuda.get_rng_state_all(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "torch_version": torch.__version__,
    }
    torch.save(ckpt, final_ckpt)
    print(f"Final ckpt: {final_ckpt.name} ({final_ckpt.stat().st_size/1e9:.2f} GB)")
    del ckpt

# Keep only the latest
ckpts = sorted(OUT_DIR.glob("ckpt_step_*.pt"))
if len(ckpts) > 1:
    for old in ckpts[:-1]:
        old.unlink()
        print(f"  deleted old: {old.name}")

manifest = f"""# MANIFEST - moshi-p4-ckpt

Phase 4 depformer co-adaptation checkpoint.

| Key | Value |
|---|---|
| step | {step} |
| phase | P4 |
| gate_metric | val_depformer_ce |
| gate_target | < 3.0 |
| init_from | mhassann/moshi-p3-ckpt/ckpt_step_1000.pt + tasfiatanha/moshi-frozen-heads |
| loss | depformer CE (8 audio codebooks, card=2048) |
| frozen | TT, in_adapter, out_adapter, emb, text_emb, out_norm, text_linear |
| trainable | depformer, depformer_in, depformer_emb, depformer_text_emb, linears |
"""
(OUT_DIR / "MANIFEST.md").write_text(manifest)

dataset_id = f"{username}/moshi-p4-ckpt"
metadata = {
    "title":    "moshi-p4-ckpt",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

print("\nFiles to upload:")
upload_files = ["MANIFEST.md", "train_log.jsonl", "dataset-metadata.json"]
upload_files += [p.name for p in OUT_DIR.glob("ckpt_step_*.pt")]
total_gb = 0.0
for name in upload_files:
    p = OUT_DIR / name
    if p.exists():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {name:<35} {gb*1000:8.1f} MB")
print(f"  {'TOTAL':<35} {total_gb:8.2f} GB")

assert total_gb < 19, f"Upload too large: {total_gb:.1f} GB"

print("\nPushing dataset ...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")
if r.returncode == 0:
    print(f"SUCCESS (create) - kaggle.com/{dataset_id}")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump ...")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", f"P4 step {step}"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
    else:
        print(f"SUCCESS (version) - kaggle.com/{dataset_id}")

print("\n=== Phase-4 session COMPLETE ===")
print(f"Step: {step}")
